# DENTRAT — Fine-Tune from Drive (Roboflow COCO)

Fine-tune your **Faster R-CNN** model on a Roboflow dataset — **no uploads**.

## Before you start (one-time on your PC)
Copy both files into **Google Drive → `dentrat_training_cache/`** (same folder as OneDrive sync):
- `dental_model_v2.pth` (or any `.pth`)
- `Dataset3.zip` or `.rar` (Roboflow COCO export)

## Colab steps
1. Runtime → **GPU (T4)**
2. **Run all cells**
3. Review class mapping in Section 4 — edit `MANUAL_OVERRIDES` if needed
4. Training ~1–2 h → model **auto-downloads** when finished (also saved to Drive)

Colab speed mode: 4000 train images, AMP, early stopping (min 5 epochs).

## 1. Install Dependencies

In [ ]:
!pip install -q albumentations opencv-python-headless scikit-learn matplotlib pandas tqdm rarfile
!apt-get -qq install -y unrar 2>/dev/null || apt-get -qq install -y unrar-free 2>/dev/null || true

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Setup + Load from Google Drive

Reads model + dataset directly from `MyDrive/dentrat_training_cache/` — **no browser upload**.

In [ ]:
import os
import copy
import time
import json
import re
import glob
import zipfile
import shutil
import random
from collections import defaultdict

import numpy as np
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader, Subset
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from google.colab import files as colab_files
from google.colab import drive

try:
    import rarfile
except ImportError:
    rarfile = None

WORK_DIR = "/content/dentrat_zip_training"
EXTRACT_DIR = os.path.join(WORK_DIR, "dataset")
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

drive.mount("/content/drive")

DRIVE_CACHE = "/content/drive/MyDrive/dentrat_training_cache"
if not os.path.isdir(DRIVE_CACHE):
    raise FileNotFoundError(
        f"Folder not found: {DRIVE_CACHE}\n"
        "Create Google Drive/dentrat_training_cache/ and copy your .pth + Dataset3.zip there."
    )


def find_drive_file(folder, extensions, prefer_names=()):
    """Pick best matching file from Drive cache folder."""
    matches = []
    for name in os.listdir(folder):
        path = os.path.join(folder, name)
        if not os.path.isfile(path):
            continue
        if os.path.splitext(name)[1].lower() in extensions:
            score = os.path.getsize(path)
            for i, prefer in enumerate(prefer_names):
                if prefer.lower() in name.lower():
                    score += 10_000_000 - i
            matches.append((score, path))
    if not matches:
        return None
    return max(matches, key=lambda x: x[0])[1]


PRETRAINED_PATH = find_drive_file(DRIVE_CACHE, {".pth"}, ("dental_model_v3", "dental_model_v2", "pretrained"))
ARCHIVE_PATH = find_drive_file(DRIVE_CACHE, {".zip", ".rar"}, ("dataset3", "dataset"))

print(f"Work dir:     {WORK_DIR}")
print(f"Drive cache:  {DRIVE_CACHE}")
print(f"Model:        {PRETRAINED_PATH or 'NOT FOUND'}")
print(f"Dataset:      {ARCHIVE_PATH or 'NOT FOUND'}")

if not PRETRAINED_PATH:
    raise FileNotFoundError(f"No .pth model in {DRIVE_CACHE}. Add dental_model_v2.pth")
if not ARCHIVE_PATH:
    raise FileNotFoundError(f"No .zip/.rar dataset in {DRIVE_CACHE}. Add Dataset3.zip")


def extract_archive(archive_path, dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    ext = os.path.splitext(archive_path)[1].lower()
    print(f"Extracting {os.path.basename(archive_path)} ...")
    if ext == ".zip":
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(dest_dir)
    elif ext == ".rar":
        if rarfile is None:
            raise ImportError("rarfile not installed")
        with rarfile.RarFile(archive_path) as rf:
            rf.extractall(dest_dir)
    else:
        raise ValueError(f"Unsupported archive: {archive_path}")


def find_dataset_root(base_dir):
    for root, dirs, _ in os.walk(base_dir):
        if "train" in dirs:
            return root
    raise FileNotFoundError("No train/ folder in archive. Export COCO format from Roboflow.")


if os.path.isdir(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
extract_archive(ARCHIVE_PATH, EXTRACT_DIR)
DATASET_DIR = find_dataset_root(EXTRACT_DIR)
print(f"Dataset root: {DATASET_DIR}")

for split in ["train", "valid", "test"]:
    split_dir = os.path.join(DATASET_DIR, split)
    if os.path.isdir(split_dir):
        n_imgs = len([f for f in os.listdir(split_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
        json_path = None
        for pat in ["_annotations.coco.json", "instances_default.json", "*.coco.json", "_annotations.json"]:
            hits = glob.glob(os.path.join(split_dir, pat))
            if hits:
                json_path = hits[0]
                break
        print(f"  {split}: {n_imgs} images, JSON: {os.path.basename(json_path) if json_path else 'NOT FOUND'}")
    else:
        print(f"  {split}: (missing)")

## 3. Load Pre-Trained Model onto GPU

In [ ]:
CLASS_NAMES = {
    1: "Caries",
    2: "Impacted Teeth",
    3: "Broken Down Crown/Root",
    4: "Infection",
    5: "Fractured Teeth",
    6: "Periodontal Bone Loss",
    7: "Other Abnormalities",
}

NUM_CLASSES = 8
IMAGE_SIZE = 416
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True


def build_model(num_classes=NUM_CLASSES):
    model = fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


def load_pretrained(model_path):
    model = build_model()
    ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
    state = ckpt.get("model_state_dict") or ckpt.get("state_dict") or ckpt
    model_dict = model.state_dict()
    pretrained = {k: v for k, v in state.items() if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(pretrained)
    model.load_state_dict(model_dict)
    print(f"Loaded {len(pretrained)}/{len(model_dict)} tensors from {os.path.basename(model_path)}")
    return model


print(f"Loading from Drive: {PRETRAINED_PATH}")
model = load_pretrained(PRETRAINED_PATH)
model.to(DEVICE)
print(f"Model ready on {DEVICE}")

## 4. Configuration & Class Mapping

In [ ]:
# moved to Section 3

In [ ]:
# dataset extracted from Drive in Section 2

## 4. Configuration & Class Mapping (continued)

In [ ]:
# ── Colab speed settings ──
BATCH_SIZE = 8
NUM_EPOCHS = 12
MAX_TRAIN_SAMPLES = 4000
USE_AMP = True
EARLY_STOP_PATIENCE = 4
MIN_EPOCHS = 5          # train at least 5 epochs before early stop can fire
MIN_VAL_IMPROVEMENT = 0.005  # ignore tiny val-loss drops (stops fake "best" at epoch 1)
FAST_AUG = True

# Higher head LR = model adapts more to new Roboflow data (fixes val loss stuck too low/flat)
LR_BACKBONE = 0.00002
LR_HEAD = 0.001
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
NUM_WORKERS = 0

OUTPUT_NAME = "dental_model_finetuned.pth"
DRIVE_OUTPUT_PATH = os.path.join(DRIVE_CACHE, OUTPUT_NAME)

KEYWORD_MAP = {
    1: ["caries", "cavity", "cavities", "decay", "dental caries"],
    2: ["impacted", "impaction", "impacted tooth", "impacted teeth"],
    3: ["broken crown", "broken down", "crown", "root", "broken-down"],
    4: ["infection", "abscess", "infected"],
    5: ["fractured", "fracture", "crack", "fractured tooth"],
    6: ["periodontal", "bone loss", "bone-loss", "periodontitis"],
    7: ["other", "abnormal", "anomaly", "misc"],
}
SKIP_KEYWORDS = ["healthy", "normal", "no finding", "background", "none"]
NUMERIC_ID_MAP = {0: None, 1: 1, 2: 2, 3: 4, 4: 5, 5: 3, 6: 6, 7: 7, 8: None}


def find_coco_json(split_dir):
    for pat in ["_annotations.coco.json", "instances_default.json", "*.coco.json", "_annotations.json"]:
        matches = glob.glob(os.path.join(split_dir, pat))
        if matches:
            return matches[0]
    return None


def load_categories(dataset_dir):
    train_dir = os.path.join(dataset_dir, "train")
    json_path = find_coco_json(train_dir)
    if not json_path:
        raise FileNotFoundError(f"No COCO JSON found in {train_dir}")
    with open(json_path) as f:
        data = json.load(f)
    return {cat["id"]: cat["name"] for cat in data.get("categories", [])}, json_path


def auto_map_category(cat_id, cat_name):
    name_lower = cat_name.lower().strip()
    for skip in SKIP_KEYWORDS:
        if skip in name_lower:
            return None
    for target_id, keywords in KEYWORD_MAP.items():
        for kw in keywords:
            if kw in name_lower or name_lower in kw:
                return target_id
    if cat_id in NUMERIC_ID_MAP:
        return NUMERIC_ID_MAP[cat_id]
    try:
        n = int(re.sub(r"\D", "", name_lower) or -1)
        if n in NUMERIC_ID_MAP:
            return NUMERIC_ID_MAP[n]
    except Exception:
        pass
    return "UNMAPPED"


raw_categories, COCO_JSON_SAMPLE = load_categories(DATASET_DIR)
print(f"Found {len(raw_categories)} categories in dataset:\n")
print(f"{'ID':<6} {'Roboflow Name':<35} {'→ Target':<25}")
print("-" * 70)

MANUAL_OVERRIDES = {}  # e.g. {9: 1, 10: None}

CATEGORY_MAP = {}
for cat_id, cat_name in sorted(raw_categories.items()):
    if cat_id in MANUAL_OVERRIDES:
        mapped = MANUAL_OVERRIDES[cat_id]
    else:
        mapped = auto_map_category(cat_id, cat_name)
    CATEGORY_MAP[cat_id] = mapped if mapped != "UNMAPPED" else None
    if mapped == "UNMAPPED":
        target_str = "⚠ UNMAPPED — add to MANUAL_OVERRIDES"
    elif mapped is None:
        target_str = "SKIP"
    else:
        target_str = f"Class {mapped}: {CLASS_NAMES.get(mapped, '?')}"
    print(f"{cat_id:<6} {cat_name:<35} {target_str}")

unmapped = [
    raw_categories[k]
    for k, v in CATEGORY_MAP.items()
    if auto_map_category(k, raw_categories[k]) == "UNMAPPED" and k not in MANUAL_OVERRIDES
]
if unmapped:
    print(f"\n⚠ Warning: {len(unmapped)} unmapped class(es): {unmapped}")
    print("Edit MANUAL_OVERRIDES above and re-run this cell before training.")
else:
    print("\n✓ All classes mapped successfully.")

## 7. COCO Dataset Loader

In [ ]:
def clip_bbox_pascal(bbox, img_w, img_h):
    xmin, ymin, xmax, ymax = bbox
    xmin = max(0.0, min(float(xmin), img_w))
    ymin = max(0.0, min(float(ymin), img_h))
    xmax = max(0.0, min(float(xmax), img_w))
    ymax = max(0.0, min(float(ymax), img_h))
    if xmax - xmin < 1.0 or ymax - ymin < 1.0:
        return None
    return [xmin, ymin, xmax, ymax]


def coco_bbox_to_pascal(bbox, img_w, img_h):
    x, y, w, h = [float(v) for v in bbox]
    if max(x, y, w, h) <= 1.0:
        x, y, w, h = x * img_w, y * img_h, w * img_w, h * img_h
    return clip_bbox_pascal([x, y, x + w, y + h], img_w, img_h)


class CocoDentalDataset(Dataset):
    def __init__(self, dataset_dir, split, category_map, transforms=None):
        self.images_dir = os.path.join(dataset_dir, split)
        self.transforms = transforms
        self.category_map = category_map

        json_path = find_coco_json(self.images_dir)
        if not json_path:
            raise FileNotFoundError(f"No COCO JSON in {self.images_dir}")

        with open(json_path) as f:
            coco = json.load(f)

        self.id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
        self.id_to_size = {img["id"]: (img["width"], img["height"]) for img in coco["images"]}
        self.annotations = defaultdict(list)
        skipped = clipped = 0

        for ann in coco["annotations"]:
            src_id = ann["category_id"]
            target = category_map.get(src_id)
            if target is None:
                skipped += 1
                continue
            img_id = ann["image_id"]
            img_w, img_h = self.id_to_size.get(img_id, (0, 0))
            if img_w <= 0 or img_h <= 0:
                continue
            bbox = coco_bbox_to_pascal(ann["bbox"], img_w, img_h)
            if bbox is None:
                clipped += 1
                continue
            self.annotations[img_id].append({"bbox": bbox, "label": target})

        self.image_ids = [iid for iid in self.id_to_file if len(self.annotations.get(iid, [])) > 0]
        print(f"  [{split}] {len(self.image_ids)} images, skipped {skipped} annotations, clipped {clipped} invalid boxes")

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        filename = self.id_to_file[img_id]
        path = os.path.join(self.images_dir, filename)
        image = cv2.imread(path)
        if image is None:
            raise FileNotFoundError(f"Could not read image: {path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        img_h, img_w = image.shape[:2]

        anns = self.annotations[img_id]
        bboxes, labels = [], []
        for a in anns:
            clipped = clip_bbox_pascal(a["bbox"], img_w, img_h)
            if clipped is not None:
                bboxes.append(clipped)
                labels.append(a["label"])

        if self.transforms and bboxes:
            t = self.transforms(image=image, bboxes=bboxes, class_labels=labels)
            image = t["image"]
            bboxes = t["bboxes"]
            labels = t["class_labels"]

        if not isinstance(image, torch.Tensor):
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0

        boxes = torch.as_tensor(bboxes, dtype=torch.float32)
        labels_t = torch.as_tensor(labels, dtype=torch.int64)
        return image, {"boxes": boxes, "labels": labels_t, "image_id": torch.tensor([idx])}


def collate_fn(batch):
    return tuple(zip(*batch))


def get_train_transforms():
    aug = [
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.RandomRotate90(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
    ]
    if not FAST_AUG:
        aug.extend([
            A.Affine(scale=(0.85, 1.15), rotate=(-30, 30), p=0.7),
            A.GaussNoise(p=0.3),
            A.ElasticTransform(alpha=1, sigma=50, p=0.3),
        ])
    aug.extend([
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])
    return A.Compose(aug, bbox_params=A.BboxParams(
        format="pascal_voc", label_fields=["class_labels"], min_visibility=0.3, clip=True,
    ))


def get_val_transforms():
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format="pascal_voc", label_fields=["class_labels"], clip=True))


def maybe_subsample(dataset, max_samples, seed=42):
    if max_samples is None or len(dataset) <= max_samples:
        return dataset
    rng = random.Random(seed)
    indices = rng.sample(range(len(dataset)), max_samples)
    print(f"  Subsampled train set: {len(dataset)} → {max_samples} images (Colab speed mode)")
    return Subset(dataset, indices)


print("Loading datasets...")
train_full = CocoDentalDataset(DATASET_DIR, "train", CATEGORY_MAP, get_train_transforms())
train_dataset = maybe_subsample(train_full, MAX_TRAIN_SAMPLES)
valid_dataset = CocoDentalDataset(DATASET_DIR, "valid", CATEGORY_MAP, get_val_transforms())

test_dir = os.path.join(DATASET_DIR, "test")
if os.path.isdir(test_dir) and find_coco_json(test_dir):
    test_dataset = CocoDentalDataset(DATASET_DIR, "test", CATEGORY_MAP, get_val_transforms())
else:
    test_dataset = valid_dataset
    print("  [test] using valid split (no test folder found)")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=NUM_WORKERS)

_est_batches = len(train_loader) * NUM_EPOCHS
_est_min = _est_batches * 1.2 / 60
print(f"\nEstimated training time: ~{_est_min:.0f} min ({len(train_loader)} batches × {NUM_EPOCHS} epochs, batch={BATCH_SIZE}, AMP={USE_AMP})")
print("\nDataLoader sanity check...")
_t0 = time.time()
_img, _tgt = train_dataset[0]
print(f"  Sample 0: image {_img.shape}, {_tgt['boxes'].shape[0]} boxes — {time.time()-_t0:.1f}s")
_t0 = time.time()
_images, _targets = next(iter(train_loader))
print(f"  First batch ({len(_images)} images) — {time.time()-_t0:.1f}s")

## 8. Fine-Tune (Colab-optimized: AMP, early stopping, ~1–2h)

Fine-tunes from your uploaded checkpoint — **not** training from scratch.

In [ ]:
CHECKPOINT_PATH = os.path.join(WORK_DIR, "dental_model_finetuned_best.pth")
FINAL_OUTPUT = os.path.join(WORK_DIR, OUTPUT_NAME)


def get_optimizer(model):
    params_backbone, params_head = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (params_backbone if "backbone" in name else params_head).append(p)
    return torch.optim.SGD([
        {"params": params_backbone, "lr": LR_BACKBONE},
        {"params": params_head, "lr": LR_HEAD},
    ], momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)


def train_one_epoch(model, loader, optimizer, epoch, scaler=None):
    model.train()
    total_loss, n_batches, skipped = 0.0, 0, 0
    use_amp = USE_AMP and DEVICE.type == "cuda"
    pbar = tqdm(loader, desc=f"Epoch {epoch} train", leave=False)
    for images, targets in pbar:
        valid = [(img, tgt) for img, tgt in zip(images, targets) if len(tgt["boxes"]) > 0]
        if not valid:
            skipped += 1
            continue
        imgs = [i.to(DEVICE, non_blocking=True) for i, _ in valid]
        tgts = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for _, t in valid]
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            loss_dict = model(imgs, tgts)
            loss = sum(loss_dict.values())
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        total_loss += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f"{loss.item():.3f}", batches=n_batches)
    avg = total_loss / max(n_batches, 1)
    print(f"  Epoch {epoch} Train Loss: {avg:.4f} (skipped {skipped} empty batches)", flush=True)
    return avg


@torch.no_grad()
def eval_loss(model, loader, desc="val"):
    model.train()  # Faster R-CNN needs train mode to compute loss with targets
    use_amp = USE_AMP and DEVICE.type == "cuda"
    total, n = 0.0, 0
    for images, targets in tqdm(loader, desc=desc, leave=False):
        valid = [(img, tgt) for img, tgt in zip(images, targets) if len(tgt["boxes"]) > 0]
        if not valid:
            continue
        imgs = [i.to(DEVICE, non_blocking=True) for i, _ in valid]
        tgts = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for _, t in valid]
        with torch.amp.autocast("cuda", enabled=use_amp):
            total += sum(model(imgs, tgts).values()).item()
        n += 1
    return total / max(n, 1)


def save_checkpoint(state_dict, epoch, val_loss, path=CHECKPOINT_PATH):
    torch.save({
        "model_state_dict": state_dict,
        "epoch": epoch,
        "val_loss": val_loss,
        "num_classes": NUM_CLASSES,
        "class_names": CLASS_NAMES,
    }, path)


def save_and_download_model(state_dict):
    checkpoint = {
        "model_state_dict": state_dict,
        "num_classes": NUM_CLASSES,
        "class_names": CLASS_NAMES,
        "category_map": CATEGORY_MAP,
        "roboflow_categories": raw_categories,
        "image_size": IMAGE_SIZE,
        "best_val_loss": best_val,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "source_model": os.path.basename(PRETRAINED_PATH),
        "dataset": os.path.basename(ARCHIVE_PATH),
    }
    torch.save(checkpoint, FINAL_OUTPUT)
    shutil.copy2(FINAL_OUTPUT, DRIVE_OUTPUT_PATH)
    print(f"Saved to Drive: {DRIVE_OUTPUT_PATH}", flush=True)
    print(f"Downloading {OUTPUT_NAME} to your computer...", flush=True)
    colab_files.download(FINAL_OUTPUT)
    print("Download started.", flush=True)


optimizer = get_optimizer(model)
scaler = torch.amp.GradScaler("cuda") if USE_AMP and DEVICE.type == "cuda" else None
train_losses, val_losses = [], []
best_val, best_state = float("inf"), None
no_improve = 0

print("Starting fine-tuning...", flush=True)
print(f"  {len(train_loader)} batches x up to {NUM_EPOCHS} epochs | head LR={LR_HEAD}", flush=True)
t0 = time.time()
for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n--- Epoch {epoch}/{NUM_EPOCHS} ---", flush=True)
    tl = train_one_epoch(model, train_loader, optimizer, epoch, scaler)
    vl = eval_loss(model, valid_loader, desc=f"Epoch {epoch} val")
    train_losses.append(tl)
    val_losses.append(vl)
    print(f"  Epoch {epoch} Val Loss: {vl:.4f}", flush=True)

    improved = vl < (best_val - MIN_VAL_IMPROVEMENT)
    if improved:
        best_val = vl
        best_state = copy.deepcopy(model.state_dict())
        save_checkpoint(best_state, epoch, best_val)
        no_improve = 0
        print(f"  New best val loss: {best_val:.4f} — checkpoint saved", flush=True)
    else:
        no_improve += 1
        if epoch >= MIN_EPOCHS:
            print(f"  No improvement ({no_improve}/{EARLY_STOP_PATIENCE})", flush=True)
            if no_improve >= EARLY_STOP_PATIENCE:
                print("  Early stopping.", flush=True)
                break
        else:
            print(f"  Warmup epoch ({epoch}/{MIN_EPOCHS}) — early stop disabled", flush=True)

print(f"\nDone in {(time.time()-t0)/60:.1f} min. Best val loss: {best_val:.4f}", flush=True)

if best_state:
    model.load_state_dict(best_state)
save_and_download_model(best_state if best_state else model.state_dict())

## 9. Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(train_losses)+1), train_losses, "b-o", label="Train", markersize=4)
ax.plot(range(1, len(val_losses)+1), val_losses, "r-o", label="Val", markersize=4)
ax.set(xlabel="Epoch", ylabel="Loss", title="Fine-Tuning Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 10. Test Evaluation — Precision / Recall / F1

In [ ]:
def box_iou(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    a1 = (b1[2]-b1[0])*(b1[3]-b1[1])
    a2 = (b2[2]-b2[0])*(b2[3]-b2[1])
    return inter / (a1+a2-inter) if (a1+a2-inter) > 0 else 0


@torch.no_grad()
def evaluate_test(model, loader, iou_thresh=0.5, conf=0.5):
    model.eval()
    tp_d, fp_d, fn_d = defaultdict(int), defaultdict(int), defaultdict(int)
    for images, targets in loader:
        outs = model([i.to(DEVICE) for i in images])
        for out, tgt in zip(outs, targets):
            gt_boxes, gt_labels = tgt["boxes"].numpy(), tgt["labels"].numpy()
            mask = out["scores"].cpu().numpy() >= conf
            preds = list(zip(
                out["boxes"].cpu().numpy()[mask],
                out["labels"].cpu().numpy()[mask],
                out["scores"].cpu().numpy()[mask],
            ))
            matched = set()
            for pb, pl, ps in sorted(preds, key=lambda x: -x[2]):
                best_iou, best_i = 0, -1
                for gi, (gb, gl) in enumerate(zip(gt_boxes, gt_labels)):
                    if gi in matched or gl != pl:
                        continue
                    iou = box_iou(pb, gb)
                    if iou > best_iou:
                        best_iou, best_i = iou, gi
                if best_iou >= iou_thresh:
                    tp_d[int(pl)] += 1
                    matched.add(best_i)
                else:
                    fp_d[int(pl)] += 1
            for gi, gl in enumerate(gt_labels):
                if gi not in matched:
                    fn_d[int(gl)] += 1

    print(f"{'Class':<30} {'Prec':>8} {'Rec':>8} {'F1':>8}")
    print("-" * 56)
    all_cls = set(list(tp_d) + list(fp_d) + list(fn_d))
    for c in sorted(all_cls):
        tp, fp, fn = tp_d[c], fp_d[c], fn_d[c]
        p = tp / (tp + fp) if tp + fp else 0
        r = tp / (tp + fn) if tp + fn else 0
        f1 = 2 * p * r / (p + r) if p + r else 0
        print(f"{CLASS_NAMES.get(c, c):<30} {p:>8.3f} {r:>8.3f} {f1:>8.3f}")


if best_state:
    model.load_state_dict(best_state)
print("\n=== Test Set Evaluation ===\n")
evaluate_test(model, test_loader)

## 11. Save & Download (backup)

Model already auto-downloaded at end of Section 8. Re-run this cell only if download failed.

In [ ]:
if os.path.isfile(FINAL_OUTPUT):
    print(f"Re-downloading {OUTPUT_NAME}...")
    colab_files.download(FINAL_OUTPUT)
else:
    checkpoint = {
        "model_state_dict": best_state if best_state else model.state_dict(),
        "num_classes": NUM_CLASSES,
        "class_names": CLASS_NAMES,
        "category_map": CATEGORY_MAP,
        "roboflow_categories": raw_categories,
        "image_size": IMAGE_SIZE,
        "best_val_loss": best_val,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "source_model": os.path.basename(PRETRAINED_PATH),
        "dataset": os.path.basename(ARCHIVE_PATH),
    }
    torch.save(checkpoint, FINAL_OUTPUT)
    colab_files.download(FINAL_OUTPUT)
print("Done — upload to models/ on Railway as dental_model_v3.pth")